# TGIS Spatial Clustering and User-Level Quality Evaluation

This notebook evaluates the performance of the spatial clustering component (hubs) in the TGIS framework at the individual user level.

### Visualizations Generated
1. **All Hubs Topology (Global & Local):** Combined 2x2 grid showing local and global hubs for all cities.
2. **User Metrics Distributions:** 
   - Combined 1x4 horizontal grid showing Coverage Percentage (CP) distribution for all cities.
   - Combined 1x4 horizontal grid showing Mean Localization Distance distribution for all cities.

In [ ]:
# ==========================================
# GLOBAL EXPERIMENT & PLOT CONFIGURATIONS
# ==========================================
# Cap all cities to this number of unique users. Set to None to run on all data.
USER_NUM = 1000

# Fixed parameters for spatial clustering
RADIUS_K = 1.0
SMOOTHING_WINDOW = 5

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import os
import glob
import time
import warnings
from sklearn.cluster import HDBSCAN, AgglomerativeClustering
import concurrent.futures

# Import parallel tracer and evaluation functions from module
from module import UniversalCascadeTracer, evaluate_clustering_quality

warnings.filterwarnings("ignore")

# Academic plot formatting (Times New Roman)
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['Times New Roman']
plt.rcParams['mathtext.fontset'] = 'custom'
plt.rcParams['mathtext.rm'] = 'Times New Roman'
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['legend.fontsize'] = 10
plt.rcParams['axes.linewidth'] = 1.2
plt.rcParams['xtick.direction'] = 'in'
plt.rcParams['ytick.direction'] = 'in'

In [ ]:
def evaluate_user_level_metrics(tracer, raw_df):
    """
    Computes coverage and mean distance for each user individually.
    - Coverage percentage: ratio of clustered coordinates (label != -1) to total valid coordinates.
    - Mean distance: average distance of clustered coordinates to their assigned local hub center.
    """
    user_results = []
    
    for uid, user_df in raw_df.groupby('uid'):
        xs = user_df['x'].values
        ys = user_df['y'].values
        
        labels = tracer.user_local_labels.get(uid, [])
        if len(labels) != len(user_df):
            continue
            
        user_hubs = tracer.user_local_hubs.get(uid, {})
        
        total_points = 0
        classified_points = 0
        distances = []
        
        for i in range(len(user_df)):
            x_val = xs[i]
            y_val = ys[i]
            if np.isnan(x_val) or np.isnan(y_val):
                continue
                
            total_points += 1
            lbl = labels[i]
            if lbl != -1:
                classified_points += 1
                if lbl in user_hubs:
                    hx, hy = user_hubs[lbl]
                    dist = ((x_val - hx)**2 + (y_val - hy)**2)**0.5
                    distances.append(dist)
                    
        cp = (classified_points / total_points) * 100.0 if total_points > 0 else 0.0
        md = np.mean(distances) if distances else 0.0
        
        user_results.append({
            'uid': uid,
            'coverage_pct': cp,
            'mean_dist_m': md,
            'total_points': total_points,
            'classified_points': classified_points
        })
        
    return pd.DataFrame(user_results)

In [ ]:
best_tracers_by_city = {}
user_metrics_by_city = {}
summary_rows = []
parquet_files = sorted(glob.glob('data/*.parquet'))
print(f"Found {len(parquet_files)} parquet files in data/")

window_sizes = [3, 5, 7]

for city_path in parquet_files:
    city_filename = os.path.basename(city_path)
    city_name = city_filename.replace('-dataset.parquet', '')
    print(f"\nProcessing {city_filename}...")
    
    # Load dataset
    df = pd.read_parquet(city_path)
    
    # Handle capping to keep runtime fast and consistent
    if USER_NUM is not None:
        print(f"Sampling first {USER_NUM} unique users for {city_name} to optimize execution...")
        unique_uids = df['uid'].unique()[:USER_NUM]
        df = df[df['uid'].isin(unique_uids)].copy()
        
    df = df.dropna(subset=['x', 'y', 'd', 't'])
    
    # Build hubs using fixed min_cluster_sizes=[15] to guarantee stability
    city_tracer = UniversalCascadeTracer(df, min_cluster_sizes=[15])
    
    # Iterate through window sizes
    for w in window_sizes:
        print(f"  Running W = {w}...")
        city_tracer.build_universal_hubs_and_log(radius_K=RADIUS_K, smoothing_window=w)
        
        metrics = evaluate_clustering_quality(city_tracer, df)
        metrics['City'] = city_name
        metrics['Window'] = w
        summary_rows.append(metrics)
        
    # Restore tracer state to W = 5 (default) for plotting downstream
    print("  Restoring tracer state to W = 5 for plotting...")
    city_tracer.build_universal_hubs_and_log(radius_K=RADIUS_K, smoothing_window=5)
    best_tracers_by_city[city_name] = city_tracer
    user_metrics_by_city[city_name] = evaluate_user_level_metrics(city_tracer, df)
    
# Build and display summary quality table
summary_df = pd.DataFrame(summary_rows)
cols = ['City', 'Window', 'Coverage_pct', 'Mean_Dist_m', 'Num_Global_Hubs', 'Num_Local_Hubs']
summary_df = summary_df[cols].round(3)

print("\n" + "="*60)
print("GLOBAL SPATIAL CLUSTERING QUALITY SUMMARY TABLE")
print("="*60)
display(summary_df)

In [ ]:
# Plot and save combined "All Hubs Topology" for all cities (2x2 grid, removed global hub text markers)
fig, axes = plt.subplots(2, 2, figsize=(10, 6.5), dpi=150)
axes = axes.flatten()
cities = list(best_tracers_by_city.keys())

for idx, city_name in enumerate(cities):
    ax = axes[idx]
    city_tracer = best_tracers_by_city[city_name]
    
    # Plot Local Hubs with brighter color (#00B4D8)
    all_l_x = []
    all_l_y = []
    for user_hubs in city_tracer.user_local_hubs.values():
        for l_coord in user_hubs.values():
            all_l_x.append(l_coord[0])
            all_l_y.append(l_coord[1])
            
    if all_l_x:
        ax.scatter(all_l_x, all_l_y, c='#00B4D8', marker='s', s=12, alpha=0.3, zorder=2, label='Local Hubs')
        
    # Plot Global Hubs (smaller size s=40, removed text markers to avoid overlap)
    g_x = [coord[0] for coord in city_tracer.global_hubs.values()]
    g_y = [coord[1] for coord in city_tracer.global_hubs.values()]
    
    if g_x:
        ax.scatter(g_x, g_y, c='red', marker='s', s=40, edgecolor='black', zorder=5, label='Global Hubs')
            
    ax.set_title(f'Global and Local Hubs in City {city_name[-1]}', weight='bold', fontsize=12)
    ax.set_xlabel('X Coordinate', fontsize=10)
    ax.set_ylabel('Y Coordinate', fontsize=10)
    ax.grid(True, linestyle=':', alpha=0.6)

# One legend at the bottom spread out
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=2, bbox_to_anchor=(0.5, 0.01), frameon=True, fontsize=11)
plt.suptitle('Global & Local Hubs Topology across Cities', fontsize=14, weight='bold', y=0.98)
plt.tight_layout(rect=[0, 0.07, 1, 0.95])

topology_path = os.path.join("figure", "hubs_topology_combined.png")
os.makedirs("figure", exist_ok=True)
plt.savefig(topology_path, dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved combined hubs topology plot to {topology_path}')

In [ ]:
# Plot and save combined "User Metrics Distribution" for all cities (wide & short, large text relative to figure)
cities = list(user_metrics_by_city.keys())

# 1. Coverage Percentage (1x4 grid, smaller figure size)
fig, axes = plt.subplots(1, 4, figsize=(11, 2.5), dpi=120, sharey=True)
for idx, city_name in enumerate(cities):
    ax = axes[idx]
    user_metrics = user_metrics_by_city[city_name]
    ax.hist(user_metrics['coverage_pct'], bins=20, color='#1D3557', edgecolor='white', alpha=0.85)
    ax.set_title(f'City {city_name[-1]}', weight='bold', fontsize=12)
    ax.set_xlabel('Coverage Percentage (CP %)', fontsize=10)
    if idx == 0:
        ax.set_ylabel('Count of Users', fontsize=10)
    ax.grid(True, linestyle=':', alpha=0.6)

plt.suptitle('User Coverage Percentage Distribution across Cities', fontsize=13, weight='bold', y=1.06)
plt.tight_layout()
cp_plot_path = os.path.join("figure", "user_coverage_distribution_combined.png")
os.makedirs("figure", exist_ok=True)
plt.savefig(cp_plot_path, dpi=300, bbox_inches='tight')
plt.show()
print(f"Saved combined coverage distribution plot to {cp_plot_path}")

# 2. Mean Localization Distance (1x4 grid, smaller figure size)
fig, axes = plt.subplots(1, 4, figsize=(11, 2.5), dpi=120, sharey=True)
for idx, city_name in enumerate(cities):
    ax = axes[idx]
    user_metrics = user_metrics_by_city[city_name]
    ax.hist(user_metrics['mean_dist_m'].dropna(), bins=20, color='#E63946', edgecolor='white', alpha=0.85)
    ax.set_title(f'City {city_name[-1]}', weight='bold', fontsize=12)
    ax.set_xlabel('Mean Distance (meters)', fontsize=10)
    if idx == 0:
        ax.set_ylabel('Count of Users', fontsize=10)
    ax.grid(True, linestyle=':', alpha=0.6)

plt.suptitle('User Mean Localization Distance Distribution across Cities', fontsize=13, weight='bold', y=1.06)
plt.tight_layout()
dist_plot_path = os.path.join("figure", "user_distance_distribution_combined.png")
os.makedirs("figure", exist_ok=True)
plt.savefig(dist_plot_path, dpi=300, bbox_inches='tight')
plt.show()
print(f"Saved combined distance distribution plot to {dist_plot_path}")

In [ ]:
def plot_qualitative_grid(tracer, target_hubs, target_d=1, save_path="qualitative_grid.png"):
    fig, axes = plt.subplots(2, 2, figsize=(14, 10), dpi=150)
    axes = axes.flatten()
    
    day_raw = tracer.raw_df[tracer.raw_df['d'] == target_d]
    
    for idx, hub_id in enumerate(target_hubs):
        ax = axes[idx]
        
        primary_info, secondary_info = tracer.trace_cascade_by_day(target_d, (1, 48), hub_id)
        
        if not day_raw.empty:
            ax.scatter(day_raw['x'], day_raw['y'], c='lightgray', s=3, alpha=0.08, zorder=1, label='All Coordinates')
            
        active_xs = []
        active_ys = []
        
        if hub_id in tracer.global_hubs:
            g_coord = tracer.global_hubs[hub_id]
            ax.scatter(g_coord[0], g_coord[1], c='black', marker='*', s=180, edgecolor='white', zorder=6, label='Trigger Global Hub')
            active_xs.append(g_coord[0])
            active_ys.append(g_coord[1])
            
        px, py = [], []
        for uid, info in primary_info.items():
            t_exp = info['t']
            h = info['hub']
            
            user_data = day_raw[(day_raw['uid'] == uid) & (day_raw['t'] >= t_exp)].sort_values('t')
            if not user_data.empty:
                ax.plot(user_data['x'], user_data['y'], color='#E63946', linestyle='-', marker='.', markersize=2, alpha=0.5, zorder=4)
                active_xs.extend(user_data['x'].tolist())
                active_ys.extend(user_data['y'].tolist())
                
            if h in tracer.global_hubs:
                px.append(tracer.global_hubs[h][0])
                py.append(tracer.global_hubs[h][1])
                active_xs.append(tracer.global_hubs[h][0])
                active_ys.append(tracer.global_hubs[h][1])
                
        if px:
            ax.scatter(px, py, c='#E63946', marker='s', s=40, zorder=5, label='Primary Impact Hubs')
            
        sx, sy = [], []
        for uid, info in secondary_info.items():
            t_exp = info['t']
            h = info['hub']
            
            user_data = day_raw[(day_raw['uid'] == uid) & (day_raw['t'] >= t_exp)].sort_values('t')
            if not user_data.empty:
                ax.plot(user_data['x'], user_data['y'], color='#FFB703', linestyle='--', marker='.', markersize=2, alpha=0.5, zorder=3)
                active_xs.extend(user_data['x'].tolist())
                active_ys.extend(user_data['y'].tolist())
                
            if h in tracer.global_hubs:
                sx.append(tracer.global_hubs[h][0])
                sy.append(tracer.global_hubs[h][1])
                active_xs.append(tracer.global_hubs[h][0])
                active_ys.append(tracer.global_hubs[h][1])
                
        if sx:
            ax.scatter(sx, sy, c='#FFB703', marker='s', s=40, edgecolor='black', linewidth=0.5, zorder=4, label='Secondary Impact Hubs')
            
        # Zoom in to active area with 15% margin and bounds aspect ratio [0.5, 2.0]
        if active_xs and active_ys:
            min_x_val = min(active_xs)
            max_x_val = max(active_xs)
            min_y_val = min(active_ys)
            max_y_val = max(active_ys)
            
            dx = max_x_val - min_x_val
            dy = max_y_val - min_y_val
            
            if dx < 1e-5: dx = 20.0
            if dy < 1e-5: dy = 20.0
            
            cx = (min_x_val + max_x_val) / 2.0
            cy = (min_y_val + max_y_val) / 2.0
            
            margin = 0.15
            width = dx * (1 + 2 * margin)
            height = dy * (1 + 2 * margin)
            
            ratio = width / height
            if ratio > 2.0:
                height = width / 2.0
            elif ratio < 0.5:
                width = height * 0.5
                
            ax.set_xlim(cx - width / 2.0, cx + width / 2.0)
            ax.set_ylim(cy - height / 2.0, cy + height / 2.0)
            
        ax.set_title(f"Impact Cascade for Centroid G{hub_id}", weight='bold', fontsize=12)
        ax.set_xlabel("X Coordinate", fontsize=10)
        ax.set_ylabel("Y Coordinate", fontsize=10)
        ax.grid(True, linestyle=':', alpha=0.5)
        
    # Create a unified figure-level legend at the bottom spread out
    handles, labels = axes[0].get_legend_handles_labels()
    by_label = dict(zip(labels, handles))
    fig.legend(by_label.values(), by_label.keys(), loc='lower center', ncol=4, bbox_to_anchor=(0.5, 0.02), frameon=True, fontsize=10)
    
    plt.suptitle("Qualitative Spatial Impact Tracing Grid", fontsize=14, weight='bold', y=0.98)
    plt.tight_layout(rect=[0, 0.06, 1, 0.95])
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()

# Find the first city and tracer
active_cities = list(best_tracers_by_city.keys())
if active_cities:
    city_name = active_cities[0]
    tracer = best_tracers_by_city[city_name]
    
    # Select top 4 global hubs by number of affected primary users on Day 1
    hub_counts = {}
    for g_hub in tracer.global_hubs.keys():
        primary_info, _ = tracer.trace_cascade_by_day(1, (1, 48), g_hub)
        if len(primary_info) > 0:
            hub_counts[g_hub] = len(primary_info)
            
    sorted_hubs = sorted(hub_counts.keys(), key=lambda h: hub_counts[h], reverse=True)
    target_hubs = sorted_hubs[:4]
    
    # Pad if necessary
    if len(target_hubs) < 4:
        all_hubs = list(tracer.global_hubs.keys())
        for h in all_hubs:
            if h not in target_hubs:
                target_hubs.append(h)
            if len(target_hubs) == 4:
                break
                
    if len(target_hubs) == 4:
        os.makedirs("figure", exist_ok=True)
        plot_qualitative_grid(tracer, target_hubs, target_d=1, save_path=os.path.join("figure", "qualitative_grid_cityA.png"))
    else:
        print("Not enough global hubs to generate a 2x2 grid.")